In [1]:
# https://www.matecdev.com/posts/landsat-sentinel-aws-s3-python.html
from pystac_client import Client
from json import load
import requests
from pyproj import Transformer
import rasterio as rio
import matplotlib.pyplot as plt
import satsearch

In [2]:
LandsatSTAC = Client.open("https://landsatlook.usgs.gov/stac-server", headers=[])

for collection in LandsatSTAC.get_collections():
    print(collection)

<CollectionClient id=landsat-c2l2-sr>
<CollectionClient id=landsat-c2l2-st>
<CollectionClient id=landsat-c2ard-st>
<CollectionClient id=landsat-c2l2alb-bt>
<CollectionClient id=landsat-c2l3-fsca>
<CollectionClient id=landsat-c2ard-bt>
<CollectionClient id=landsat-c2l1>
<CollectionClient id=landsat-c2l3-ba>
<CollectionClient id=landsat-c2l2alb-st>
<CollectionClient id=landsat-c2ard-sr>
<CollectionClient id=landsat-c2l2alb-sr>
<CollectionClient id=landsat-c2l2alb-ta>
<CollectionClient id=landsat-c2l3-dswe>
<CollectionClient id=landsat-c2ard-ta>


In [3]:
def BuildSquare(lon, lat, delta):
    c1 = [lon + delta, lat + delta]
    c2 = [lon + delta, lat - delta]
    c3 = [lon - delta, lat - delta]
    c4 = [lon - delta, lat + delta]
    geometry = {"type": "Polygon", "coordinates": [[ c1, c2, c3, c4, c1 ]]}
    return geometry

geometry = BuildSquare(-59.346271, -34.233076, 0.04)
timeRange = '2019-06-01/2021-06-01'

In [4]:
LandsatSearch = LandsatSTAC.search ( 
    intersects = geometry,
    datetime = timeRange,
    query =  ['eo:cloud_cover95'],
    collections = ["landsat-c2l2-sr"] )

Landsat_items = [i.to_dict() for i in LandsatSearch.items()]
print(f"{len(Landsat_items)} Landsat scenes fetched")

193 Landsat scenes fetched


In [7]:

print(Landsat_items[0]['assets']['red']['href'])    
print(Landsat_items[0]['assets']['red']['alternate']['s3']['href'])

https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF
s3://usgs-landsat/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF


In [8]:
def download_landsat(landsat_url, download_path):
    response = requests.get(landsat_url, stream=True)
    if response.status_code == 200:
        with open(download_path, 'wb') as f:
            for chunk in response.iter_content(1024):
                f.write(chunk)
    else:
        raise Exception(f"Failed to download Landsat data. Status code: {response.status_code}")

In [9]:
from pyproj import Transformer

def getSubset(geotiff_file, bbox):
    with rio.open(geotiff_file) as geo_fp:
        # Calculate pixels with PyProj
        Transf = Transformer.from_crs("epsg:4326", geo_fp.crs)
        lat_north, lon_west = Transf.transform(bbox[3], bbox[0])
        lat_south, lon_east = Transf.transform(bbox[1], bbox[2])
        x_top, y_top = geo_fp.index(lat_north, lon_west)
        x_bottom, y_bottom = geo_fp.index(lat_south, lon_east)
        
        # Define window in RasterIO
        window = rio.windows.Window.from_slices((x_top, x_bottom), (y_top, y_bottom))
        
        # Read the subset
        subset = geo_fp.read(1, window=window)
    
    return subset


In [10]:
def plotNDVI(nir,red,filename):
    ndvi = (nir-red)/(nir+red)
    ndvi[ndvi>1] = 1
    plt.imshow(ndvi)
    plt.savefig(filename)
    plt.close()

In [12]:
from rasterio.features import bounds
import matplotlib.pyplot as plt
import os

bbox = bounds(geometry)


for i,item in enumerate(Landsat_items):
    red_href = item['assets']['red']['href']
    nir_href =  item['assets']['nir08']['href']
    date = item['properties']['datetime'][0:10]
    red_path = os.path.join("./", f"{date}_red.tif")
    nir_path = os.path.join("./", f"{date}_nir.tif")
    download_landsat(red_href, red_path)
    download_landsat(nir_href, nir_path)

    print("Landsat item number " + str(i) + "/" + str(len(Landsat_items)) + " " + date)
    red = getSubset(red_path, bbox)
    nir = getSubset(nir_path, bbox)
    plotNDVI(nir,red,"landsat/" + date + "_ndvi.png")


Landsat item number 0/193 2021-05-28


RasterioIOError: './2021-05-28_red.tif' not recognized as being in a supported file format.

In [ ]:
import requests
import os

# Your USGS Authentication Token (replace with your actual token)
auth_token = "cs5131"

# Function to download the file using the auth token
def download_file(url, save_path):
    headers = {
        'Authorization': f'Bearer {auth_token}'
    }
    
    # Make the request to download the file
    response = requests.get(url, headers=headers, stream=True)
    
    if response.status_code == 200:
        with open(save_path, 'wb') as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        print(f"Downloaded: {save_path}")
    else:
        print(f"Failed to download {url}. Status Code: {response.status_code}")

# Example usage (for the Red band from LandsatLook)
red_url = "https://landsatlook.usgs.gov/data/collection02/level-2/standard/oli-tirs/2021/225/084/LC08_L2SP_225084_20210528_20210607_02_T1/LC08_L2SP_225084_20210528_20210607_02_T1_SR_B4.TIF"
save_path = os.path.join("./", "LC08_L2SP_20210528_B4.TIF")

# Download the Red band
download_file(red_url, save_path)


Downloaded: ./LC08_L2SP_20210528_B4.TIF


In [15]:
import requests

API_KEY = "your_usgs_api_key"

# Set the base URL
base_url = "https://m2m.cr.usgs.gov/api/api/json/stable/scene-search"


# Use your API key to authenticate
headers = {
    "X-Auth-Token": "AKqfqOHSiydilBG@ln2eNXXuY1QSfgdKMPFE6_!J_p068fgWtvcRdr679kUkprg1"
}
search_url = base_url + "scene-search"

search_payload = {
    "datasetName": "LANDSAT_8_C1",
    "spatialFilter": {
        "filterType": "mbr",  # minimum bounding rectangle
        "lowerLeft": {"latitude": 37.90, "longitude": -122.30},
        "upperRight": {"latitude": 37.92, "longitude": -122.28}
    },
    "temporalFilter": {
        "startDate": "2021-06-01",
        "endDate": "2021-06-30"
    },
    "maxResults": 5,
    "sortOrder": "DESC"
}

response = requests.post(search_url, json=search_payload, headers=headers)
scenes = response.text
print(scenes)


{"requestId": 0, "version": "stable", "data": null, "errorCode": "AUTH_KEY_INVALID", "errorMessage": "Invalid API Key"}


In [ ]:
download_url = base_url + "download-request"

download_payload = {
    "datasetName": "LANDSAT_8_C1",
    "products": ["STANDARD"],  # Could be "STANDARD", "SR", etc.
    "entityIds": [scene["entityId"] for scene in scenes]
}

response = requests.post(download_url, json=download_payload, headers=headers)
download_links = response.json()['data']['availableDownloads']


In [ ]:
for download in download_links:
    url = download['url']
    filename = url.split("/")[-1]

    r = requests.get(url, stream=True)
    with open(filename, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)

    print(f"Downloaded: {filename}")
